# 04 - Build Africa Level 2 Graph Variants and Train Residual GNNs

In [ ]:
from pathlib import Path
import sys
# Resolve local package imports from common notebook launch locations.
ROOT = Path.cwd() if (Path.cwd() / 'src').exists() else Path.cwd().parent
sys.path.append(str(ROOT / 'src'))

import pandas as pd

from grace_gnn.config import DATA_RAW, EXPERIMENT_REGION, LAGGED_DATASET_CSV, REGION_OUTPUTS, REGION_PREDICTIONS_CSV, AFRICA_L2_NO_MADAGASCAR_BASIN_NAMES, TRAIN_FRACTION, VAL_FRACTION, TEST_FRACTION, RANDOM_SEED, ensure_dirs
from grace_gnn.data import find_mask_zips
from grace_gnn.features import feature_columns, filter_region
from grace_gnn.graph import build_knn_edges_from_mask_zips, make_degree_matched_random_edges, reverse_edges, save_edges, symmetrize_edges
from grace_gnn.models import train_manual_gcn, torch_available
from grace_gnn.splits import chronological_fraction_split
from grace_gnn.evaluate import graph_prediction_frame

ensure_dirs()
# Leave basin filtering open unless a focused mask subset is needed.
MASK_BASIN_NAME_FILTER = None

In [ ]:
# Load the same regional lagged table used by the baseline notebooks.
if not LAGGED_DATASET_CSV.exists():
    print(f'Missing {LAGGED_DATASET_CSV}. Run notebook 02 first.')
    lagged = None
else:
    lagged = pd.read_csv(LAGGED_DATASET_CSV, parse_dates=['date'])
    lagged = filter_region(lagged, EXPERIMENT_REGION)
    print(f'Loaded {len(lagged):,} Africa Level 2 lagged samples across {lagged["basin_id"].nunique()} regions')

In [ ]:
if lagged is not None:
    # Mask zips provide basin geometry for constructing spatial graph edges.
    mask_zips = find_mask_zips(DATA_RAW) + find_mask_zips(ROOT / 'masks')
    if not mask_zips:
        print('No mask zips available; skipping GNN training.')
    elif not torch_available():
        print('PyTorch is unavailable; skipping GNN training.')
    else:
        basin_ids = sorted(lagged['basin_id'].astype(str).unique())
        # Use the available basin IDs to keep random graph variants on the same node set.
        real_directed = build_knn_edges_from_mask_zips(
            mask_zips,
            AFRICA_L2_NO_MADAGASCAR_BASIN_NAMES,
            REGION_OUTPUTS / 'edges_real_knn_directed.csv',
            k=3,
            graph_type='real_knn_directed',
        )
        if real_directed.empty:
            print('No within-region source edges found; using undirected complete proximity fallback is not allowed for this experiment.')
        else:
            graph_variants = {
            # Compare real directed edges against controlled graph transformations.
                'real_knn_directed': real_directed,
                'real_knn_undirected': symmetrize_edges(real_directed, graph_type='real_knn_undirected'),
                'real_knn_reversed': reverse_edges(real_directed, graph_type='real_knn_reversed'),
                'random_degree_matched': make_degree_matched_random_edges(real_directed, basin_ids, seed=RANDOM_SEED),
            }
            for graph_type, edges in graph_variants.items():
                path = REGION_OUTPUTS / f'edges_{graph_type}.csv'
            # Persist edge lists so evaluation and plotting can inspect each graph.
                save_edges(edges, path)
                print(f'Saved {len(edges):,} {graph_type} edges to {path}')
            splits = chronological_fraction_split(lagged, TRAIN_FRACTION, VAL_FRACTION, TEST_FRACTION)
            features = feature_columns(lagged)
            prediction_parts = []
            # Train one residual GCN per graph variant on identical chronological splits.
            for graph_type, edges in graph_variants.items():
                _, preds = train_manual_gcn(
                    splits['train'], splits['val'], splits['test'], features, edges, basin_ids,
                    seed=RANDOM_SEED, residual=True
                )
                for split_name, pred_df in preds.items():
                    prediction_parts.append(graph_prediction_frame(pred_df, 'residual_neighbor_gnn', graph_type, split_name))
            gnn_predictions = pd.concat(prediction_parts, ignore_index=True)
            existing = pd.read_csv(REGION_PREDICTIONS_CSV, parse_dates=['date']) if REGION_PREDICTIONS_CSV.exists() else pd.DataFrame()
            if not existing.empty:
            # Merge GNN predictions with existing non-GNN predictions without duplicating runs.
                existing['basin_id'] = existing['basin_id'].astype(str)
                existing = existing[existing['model_name'] != 'residual_neighbor_gnn']
            gnn_predictions['basin_id'] = gnn_predictions['basin_id'].astype(str)
            combined = pd.concat([existing, gnn_predictions], ignore_index=True)
            combined = combined.drop_duplicates(['date', 'basin_id', 'model_name', 'graph_type', 'split'], keep='last')
            REGION_PREDICTIONS_CSV.parent.mkdir(parents=True, exist_ok=True)
            combined.to_csv(REGION_PREDICTIONS_CSV, index=False)
            print(f'Saved combined regional predictions to {REGION_PREDICTIONS_CSV}')
            display(gnn_predictions.head())